# 01 · Left snapshot counts → S3
Run daily where the Left databases are reachable (cron / Task Scheduler / VS Code).

In [ ]:
# pairs.json, shared by 01 (left, on-prem) and 02 (Databricks); 01 uploads it next to the counts
#
# pairs.<name>.left
#   source       "oracle" (env <conn>_USR / _PWD / _DSN) | "duckdb" (path)
#   conn | path  connection name, or the .duckdb file
#   table | query, where (optional SQL filter)
#   as_of        {"column": "EFF_DT"}   -> MAX(day of EFF_DT)   (view carries its own date)
#                {"offset_days": 0|1}   -> run date - n         (view = today / yesterday)
#   count        "all" (default, the whole current view) | "as_of" (only rows on the as-of day)
# pairs.<name>.right
#   table        catalog.schema.table
#   date_col     default dw_bus_dt;  date_type: date (default) | string ('YYYY-MM-DD') | int (YYYYMMDD)
#   offset_days  right day = as_of + n (default 0);  where (optional SQL filter)
# pairs.<name> (optional, copied into the generated dtrack config)
#   col_map      partial {LEFT: right}; the rest is matched by name, case-insensitive
#   key, key_families, sample, vintage, strict, col_type_overrides

CONFIG = "pairs.json"
S3_PREFIX = "s3://<bucket>/<prefix>/count_watch"   # a Databricks Volume must point at this prefix
RUN_DATE = None                                    # None = today in the config timezone; "YYYY-MM-DD" to backfill
PAIRS = None                                       # e.g. ["accounts"]
DRY_RUN = False                                    # True: count, print, upload nothing

In [ ]:
import json
import os
from datetime import UTC, date, datetime, timedelta
from pathlib import Path
from urllib.parse import urlparse
from zoneinfo import ZoneInfo

import boto3
import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
config = json.loads(Path(CONFIG).read_text())
zone = ZoneInfo(config.get("timezone", "America/New_York"))
run_date = date.fromisoformat(RUN_DATE) if RUN_DATE else datetime.now(zone).date()
bucket, prefix = urlparse(S3_PREFIX).netloc, urlparse(S3_PREFIX).path.strip("/")
s3 = boto3.client("s3")
print("run date:", run_date, "| pairs:", list(config["pairs"]) if not PAIRS else PAIRS)

In [ ]:
def connect(left):
    source = left.get("source", "oracle")
    if source == "oracle":
        import oracledb

        name = left["conn"]
        return oracledb.connect(
            user=os.environ[f"{name}_USR"], password=os.environ[f"{name}_PWD"], dsn=os.environ[f"{name}_DSN"]
        )
    if source == "duckdb":
        import duckdb

        return duckdb.connect(left["path"], read_only=True)
    raise ValueError(f"unsupported left source {source!r}")


def run_sql(left, sql):
    connection = connect(left)
    try:
        cursor = connection.cursor()
        cursor.execute(sql)
        columns = [item[0] for item in cursor.description or []]
        return columns, cursor.fetchall()
    finally:
        connection.close()


def day(left, column):
    return f"TRUNC({column})" if left.get("source", "oracle") == "oracle" else f"CAST({column} AS DATE)"


def relation(left):
    return f"({left['query']}) q" if left.get("query") else left["table"]


def where(*clauses):
    kept = [f"({c})" for c in clauses if c]
    return f" WHERE {' AND '.join(kept)}" if kept else ""


def iso(value):
    return value.date().isoformat() if isinstance(value, datetime) else date.fromisoformat(str(value)[:10]).isoformat()

In [ ]:
def snapshot(name, pair):
    left = pair["left"]
    rule = left.get("as_of") or {"offset_days": 0}
    if "column" in rule:
        _, rows = run_sql(left, f"SELECT MAX({day(left, rule['column'])}) FROM {relation(left)}{where(left.get('where'))}")
        if rows[0][0] is None:
            raise ValueError(f"{name}: {rule['column']} is empty")
        as_of = iso(rows[0][0])
    else:
        as_of = (run_date - timedelta(days=int(rule["offset_days"]))).isoformat()
    on_day = None
    if left.get("count", "all") == "as_of":
        on_day = f"{day(left, rule['column'])} = DATE '{as_of}'"     # needs as_of.column
    _, rows = run_sql(left, f"SELECT COUNT(*) FROM {relation(left)}{where(left.get('where'), on_day)}")
    columns, _ = run_sql(left, f"SELECT * FROM {relation(left)} WHERE 1=0")
    return {
        "pair": name,
        "as_of": as_of,
        "left_count": int(rows[0][0]),
        "columns": columns,
        "count_mode": left.get("count", "all"),
        "as_of_rule": rule,
        "run_date": run_date.isoformat(),
        "pulled_at": datetime.now(UTC).isoformat(timespec="seconds"),
    }


records, errors = [], {}
for name, pair in config["pairs"].items():
    if PAIRS and name not in PAIRS:
        continue
    try:
        records.append(snapshot(name, pair))
    except Exception as exc:                       # one broken pair must not stop the others
        errors[name] = f"{type(exc).__name__}: {exc}"
display(pd.DataFrame(records)[["pair", "as_of", "left_count", "count_mode", "pulled_at"]] if records else "no records")
errors

In [ ]:
# layout read by 02:  <prefix>/pairs.json  and  <prefix>/left/<pair>/<as_of>.json (re-runs overwrite)
def put(key, value):
    s3.put_object(Bucket=bucket, Key=f"{prefix}/{key}", Body=json.dumps(value, indent=2).encode(), ContentType="application/json")
    return f"s3://{bucket}/{prefix}/{key}"


if DRY_RUN:
    print("DRY_RUN: nothing uploaded")
else:
    print(put("pairs.json", config))
    for record in records:
        print(put(f"left/{record['pair']}/{record['as_of']}.json", record))
assert not errors, f"failed pairs: {sorted(errors)}"